# 01 · ColVec over the full corpus (Arm A and the visual half of Arm B)

Encodes **every page of every PDF** in `0.1. BENCHMARK` with `webAI-ColVec1.1-8b` and scores it against **all
queries**. The arms then apply any gate, at any `k`, locally with no GPU, so this runs once.

**Input:** `colvec_bundle.zip` from `python research/experiments/prep_ondemand_bundle.py --target colvec`, put in your Drive folder once.
**Output:** in `<Drive folder>/<fingerprint>/colvec/`: `colvec_result.zip` for the local scripts, and `PASTE_ME.md` to paste into the chat.
The last cell also prints `PASTE_ME.md`.

| | |
|---|---|
| GPU | Colab Pro **L4 24 GB** (8b in bf16 uses ~17 GB). A100 is faster. Not a T4. |
| Time | about **1.3 s per page**, unbatched, so roughly 1.8 h for 5,091 pages. |
| Checkpoint | every 250 pages, copied to Drive with the timing log. The checkpoint is only the score columns done so far (a few MB), so a disconnect costs minutes. |
| Resume | rerun the notebook from the top. The full-run cell finds the newest checkpoint on Drive by itself. |

**Why scores are computed per page instead of storing embeddings.** At 1792 visual tokens x 640 dims x fp16, one page is
about 2.3 MB, so 5,091 pages is **about 11-12 GB**. The 220 queries are fixed, so each page's 220 scores are computed the
moment it is encoded and the embedding is dropped. Raw embeddings are an opt-in extra (`SAVE_EMBEDDINGS`), sharded to ~1 GB files.

**Sanity check built in.** The bundle carries the score matrix from your earlier 5,334-page run. Before the long run, the
notebook re-scores 12 pages and compares. If this environment does not reproduce the old scores it **stops**, instead of
spending two hours on a drifted install.

In [ ]:
!nvidia-smi

## 1 · Connect Drive, get the bundle, verify it

Drive is where everything durable goes: checkpoints, the result, and `PASTE_ME.md`. The run itself works on the local disk
and copies to Drive at each checkpoint, because Drive is a network mount that is slow for many small writes and can drop.
Put `colvec_bundle.zip` in the Drive folder once and every later session reads it from there. If it is not there, you are asked to upload it.

In [ ]:
from google.colab import files
import hashlib, json, os, shutil, time, zipfile
from pathlib import Path

USE_DRIVE    = True
DRIVE_FOLDER = "FOR-COLAB/Colab-for-Axiom-DE-RnD"    # path under My Drive (colab/README.md, "Drive setup")
BUNDLE_NAME  = "colvec_bundle.zip"

REPO = Path("/content/AXIOM_DE-RD"); REPO.mkdir(exist_ok=True)
BENCH = REPO / "0.1. BENCHMARK"

DRIVE = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    if not DRIVE.is_dir():
        raise SystemExit(f"no folder {DRIVE_FOLDER!r} in My Drive. It has: {sorted(os.listdir('/content/drive/MyDrive'))}. "
                         "Set DRIVE_FOLDER to its path under My Drive. If the folders you expect are not listed at all, Colab mounted "
                         "a different Google account: run drive.flush_and_unmount(), then rerun this cell and pick the account that owns the folder.")

bundle_zip = Path("/content") / BUNDLE_NAME
if DRIVE and (DRIVE / BUNDLE_NAME).exists():
    shutil.copy(DRIVE / BUNDLE_NAME, bundle_zip)       # unzip locally: 103 PDFs written onto Drive would be minutes of churn
    print("bundle read from Drive")
else:
    uploaded = files.upload()                          # choose colvec_bundle.zip
    name = next(n for n in uploaded if n.endswith(".zip")); del uploaded
    if Path(name).resolve() != bundle_zip.resolve():
        shutil.move(name, bundle_zip)
    if DRIVE:
        shutil.copy(bundle_zip, DRIVE / BUNDLE_NAME); print("bundle saved to Drive for the next session")
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall(REPO)

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

manifest = json.loads((REPO / "BUNDLE_MANIFEST.json").read_text())
assert manifest["target"] == "colvec", f"this notebook needs a colvec bundle, got {manifest['target']!r}"
bad = [rel for rel, digest in manifest["files"].items() if _sha256(REPO / rel) != digest]
assert not bad, f"{len(bad)} files are corrupt or missing (truncated upload?), e.g. {bad[:3]}"

h = hashlib.sha256()
for name in ("documents.jsonl", "queries.jsonl", "qrels.jsonl"):
    h.update(name.encode()); h.update((BENCH / name).read_bytes())
FINGERPRINT = h.hexdigest()[:12]
assert FINGERPRINT == manifest["bundle_fingerprint"], "manifests do not match the bundle fingerprint"

documents = [json.loads(l) for l in (BENCH / "documents.jsonl").read_text().splitlines() if l.strip()]
queries = [json.loads(l) for l in (BENCH / "queries.jsonl").read_text().splitlines() if l.strip()]
print(f"bundle {FINGERPRINT}{'  (SMOKE)' if manifest['smoke'] else ''}: {len(documents)} documents, "
      f"{manifest['n_pages']} pages, {len(queries)} queries, {len(manifest['files'])} files verified")
print("code from git", manifest["git"]["sha"][:8], f"with {manifest['git']['uncommitted_files']} uncommitted files")

# One folder per bundle and notebook, so a smoke run and the full run never share checkpoints.
RUN = (DRIVE / FINGERPRINT / "colvec") if DRIVE else Path("/content/colvec_run")
RUN.mkdir(parents=True, exist_ok=True)

def to_drive(path, folder=None):
    """Copy a file into the run folder. A Drive error is printed, never raised: the local copy is still there."""
    dest = (folder or RUN) / Path(path).name
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        part = dest.with_name(dest.name + ".part")
        shutil.copy(path, part); part.replace(dest)
        return dest
    except OSError as e:
        part.unlink(missing_ok=True)
        print(f"  could not copy {Path(path).name} to {dest.parent}: {e}. The local copy is kept.")

if DRIVE:                                  # prove a write + rename + read works now, not two hours in
    probe = Path("/content/drive_probe.txt"); probe.write_text(FINGERPRINT)
    copied = to_drive(probe)
    if not copied or copied.read_text() != FINGERPRINT:
        raise SystemExit("writing to Drive does not work (see the message above). Fix it, or set USE_DRIVE = False to use browser downloads.")
    copied.unlink()
print("run folder:", RUN)

## 2 · Install

Same pins as the notebooks that already worked, without the editable repo install they carried. `config.json` of webAI-ColVec pins transformers 5.14.1
(`model_type: qwen3_5`), and these three pins come from the repo's `evaluation-requirements-cu128.txt`.
If pip warns about a conflict: **Runtime > Restart session**, then continue from cell 3. Files under `/content` survive a restart.

In [ ]:
%cd /content/AXIOM_DE-RD
# No `pip install -e .` here. This notebook imports no repo code, and the editable install pulls ~30 packages
# (boto3, streamlit, pytrec_eval-terrier which compiles from source, ...) whose pins have collided with Colab's
# preinstalled numpy/pandas before ("pip's dependency resolver does not currently take into account all the packages").
%pip install -q pymupdf
%pip install -q -U "transformers==5.14.1" "sentence-transformers==5.6.0" "accelerate==1.14.0"
%pip install -q -U bitsandbytes

## 3 · Config

In [ ]:
import torch
from importlib.metadata import version

MODEL_ID          = "webAI-Official/webAI-ColVec1.1-8b"   # "...-4b" fits a T4, but is not the model the earlier numbers used
MAX_VISUAL_TOKENS = 1792       # the earlier runs used this. Changing it changes every score.
DPI               = 144
LOAD_8BIT         = False
RESUME            = True       # continue from the newest checkpoint in the run folder, if there is one
SAVE_EMBEDDINGS   = False      # True: keep raw page embeddings in ~1 GB shards (about 11 GB on Drive). Off: scores only.
SHARD_BYTES       = 1_000_000_000
CKPT_EVERY        = 250        # checkpoint, and copy it to Drive
PROBE_PAGES       = 12
FORCE             = False      # True: skip the reproduction gate (not recommended)

DEVICE = "cuda"
gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_memory / 1024**3
DTYPE = torch.bfloat16 if any(x in gpu.name for x in ("A100", "H100", "L4", "L40", "RTX 40")) else torch.float16
if "8b" in MODEL_ID and vram_gb < 20 and not LOAD_8BIT:
    raise SystemExit(f"{gpu.name} has {vram_gb:.0f} GiB: use an L4/A100, or LOAD_8BIT=True (slower, and not the same numerics)")
ENV = {k: version(k) for k in ("torch", "transformers", "accelerate", "sentence-transformers", "pymupdf", "numpy")}
print(f"GPU {gpu.name} {vram_gb:.0f} GiB | dtype {DTYPE} | {ENV}")

# Colab's pip can silently keep its preinstalled copy of a package instead of moving to the pin (this cost a whole
# ViSAR session: the model loaded, then failed on an adapter, with no install error). Check the result, not the log.
PINS = {"transformers": "5.14.1", "sentence-transformers": "5.6.0", "accelerate": "1.14.0"}
wrong = {k: {"installed": ENV[k], "pin": v} for k, v in PINS.items() if ENV[k] != v}
if wrong:
    raise SystemExit(f"installed versions differ from the pins: {wrong}. Runtime > Restart session, rerun the install cell, then this one. "
                     "Do not pin numpy: Colab's scipy/pandas are built against numpy 2, and 1.x breaks their C ABI.")

## 4 · Load the model, encode the queries

In [ ]:
import gc, numpy as np
from time import perf_counter
from transformers import AutoModel, AutoProcessor

TIMINGS = Path("/content/timings.jsonl"); TIMINGS.write_text("")
def log_timing(stage, unit, n, seconds, **extra):
    row = {"stage": stage, "unit": unit, "n": n, "seconds": round(seconds, 6), "bundle": FINGERPRINT,
           "gpu": gpu.name, "model": MODEL_ID, "ts": time.strftime("%Y-%m-%dT%H:%M:%S"), **extra}
    with TIMINGS.open("a") as f:
        f.write(json.dumps(row) + "\n")

if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()

_load_kw = dict(trust_remote_code=True, attn_implementation="sdpa", device_map="cuda:0")
if LOAD_8BIT:
    from transformers import BitsAndBytesConfig
    _load_kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    _load_kw["dtype"] = DTYPE

t = perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_num_visual_tokens=MAX_VISUAL_TOKENS)
model = AutoModel.from_pretrained(MODEL_ID, **_load_kw).eval()
log_timing("colvec_model_load", "model", 1, perf_counter() - t)
print(f"model loaded in {perf_counter() - t:.0f}s: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B params")

def embed(inp):
    out = model(**inp)
    if hasattr(out, "embeddings"):
        out = out.embeddings
    elif hasattr(out, "last_hidden_state"):
        out = out.last_hidden_state
    elif isinstance(out, (tuple, list)):
        out = out[0]
    return out

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

In [ ]:
qids = [q["query_id"] for q in queries]
QB = 8
query_emb = {}
t = perf_counter()
for s in range(0, len(queries), QB):
    chunk = queries[s:s + QB]
    inp = processor.process_queries([q["query"] for q in chunk]).to(model.device)
    with torch.inference_mode():
        emb = embed(inp)
    for q, e in zip(chunk, emb):
        query_emb[q["query_id"]] = e.to(torch.float16).cpu()
sync(); log_timing("colvec_query_encode", "query", len(queries), perf_counter() - t)

# Padded once and kept on the GPU, so every page is scored the moment it is encoded.
Q = torch.nn.utils.rnn.pad_sequence([query_emb[i] for i in qids], batch_first=True).to(DEVICE, torch.float16)
qmask = torch.nn.utils.rnn.pad_sequence([torch.ones(query_emb[i].shape[0]) for i in qids], batch_first=True).to(DEVICE)
DIM = int(Q.shape[-1])
torch.save(query_emb, "/content/query_embeddings.pt")
print(f"{len(qids)} queries encoded in {perf_counter() - t:.1f}s | Q {tuple(Q.shape)} dim {DIM}")

## 5 · Encode and score pages

`encode_page` is the whole per-page path. It is also what the reproduction probe below runs, so the probe tests exactly
what the long run will do. Batch size is 1, as in the proven recipe, so the per-page timings are the true unbatched
on-demand cost.

In [ ]:
import fitz
from PIL import Image

def render(pdf, page0):
    pix = pdf[page0].get_pixmap(dpi=DPI, colorspace=fitz.csRGB, alpha=False)
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

def encode_page(pdf, page0):
    """(score column over all queries as float32, token count, timing dict)."""
    t0 = perf_counter()
    img = render(pdf, page0)
    t1 = perf_counter()
    inp = processor.process_images([img]).to(model.device)
    with torch.inference_mode():
        e = embed(inp)[0].to(torch.float16)
        sync(); t2 = perf_counter()
        sim = torch.einsum("nld,md->nlm", Q, e)
        col = (sim.max(dim=2).values * qmask).sum(dim=1).float()
        sync(); t3 = perf_counter()
    return col.cpu().numpy(), e, {"render_seconds": t1 - t0, "encode_seconds": t2 - t1, "score_seconds": t3 - t2,
                                  "tokens": int(e.shape[0])}

page_list = [(f"{d['doc_id']}#page={p}", d, p) for d in documents for p in range(d["metadata"]["page_count"])]
assert len(page_list) == manifest["n_pages"]
page_keys = [k for k, _, _ in page_list]
col_of = {k: i for i, k in enumerate(page_keys)}
for d in documents:            # the manifest's page count must be the PDF's, or the unit ids are wrong
    with fitz.open(BENCH / d["path"]) as pdf:
        assert len(pdf) == d["metadata"]["page_count"], f"{d['doc_id']}: {len(pdf)} pages, manifest says {d['metadata']['page_count']}"
print(f"{len(page_keys)} pages across {len(documents)} PDFs")

## 6 · Reproduction gate: re-score a few pages and compare to your earlier run

In [ ]:
REF = REPO / "reference"

def compare_to_reference(new_scores, new_keys, new_qids):
    """Agreement with the earlier export on the (query, page) cells both cover. None if there is no overlap."""
    if not (REF / "colvec_scores.npy").exists():
        return None
    ref = np.load(REF / "colvec_scores.npy")
    rkeys = json.loads((REF / "colvec_keys.json").read_text()); rq = json.loads((REF / "colvec_qids.json").read_text())
    rk_i, rq_i = {k: i for i, k in enumerate(rkeys)}, {q: i for i, q in enumerate(rq)}
    ki = [(j, rk_i[k]) for j, k in enumerate(new_keys) if k in rk_i]
    qi = [(j, rq_i[q]) for j, q in enumerate(new_qids) if q in rq_i]
    if not ki or not qi:
        return None
    A = new_scores[np.ix_([a for a, _ in qi], [a for a, _ in ki])]
    B = ref[np.ix_([b for _, b in qi], [b for _, b in ki])]
    return {"n_queries": len(qi), "n_pages": len(ki),
            "pearson": float(np.corrcoef(A.ravel(), B.ravel())[0, 1]),
            "max_abs_diff": float(np.abs(A - B).max()), "mean_abs_diff": float(np.abs(A - B).mean()),
            "score_scale": float(np.abs(B).mean()),
            "top1_agreement": float((A.argmax(1) == B.argmax(1)).mean())}

# Probe pages that the earlier run also encoded (ViDoRe docs share ids), spread across documents.
ref_keys = set(json.loads((REF / "colvec_keys.json").read_text())) if (REF / "colvec_keys.json").exists() else set()
overlap = [(k, d, p) for k, d, p in page_list if k in ref_keys]
step = max(len(overlap) // PROBE_PAGES, 1)
probe = overlap[::step][:PROBE_PAGES]
probe_scores, probe_keys, per_page = [], [], []
for key, d, p in probe:
    with fitz.open(BENCH / d["path"]) as pdf:
        col, _, tm = encode_page(pdf, p)
    probe_scores.append(col); probe_keys.append(key); per_page.append(tm)

if probe:
    est = np.mean([sum(x.values()) - x["tokens"] for x in per_page])
    print(f"per page: {est:.2f}s (render {np.mean([x['render_seconds'] for x in per_page]):.2f} + "
          f"encode {np.mean([x['encode_seconds'] for x in per_page]):.2f} + score {np.mean([x['score_seconds'] for x in per_page]):.3f}) "
          f"-> about {est * len(page_keys) / 3600:.1f} h for {len(page_keys)} pages | "
          f"peak GPU {torch.cuda.max_memory_allocated() / 1024**3:.1f} GiB | tokens/page {np.mean([x['tokens'] for x in per_page]):.0f}")
cmp = compare_to_reference(np.stack(probe_scores, 1), probe_keys, qids) if probe else None
if cmp is None:
    REPRO_OK = FORCE
    print("no earlier scores to compare with (or no overlapping pages). Set FORCE = True to continue anyway.")
else:
    REPRO_OK = FORCE or (cmp["pearson"] >= 0.999 and cmp["top1_agreement"] >= 0.9)
    print(f"vs earlier run over {cmp['n_queries']} queries x {cmp['n_pages']} pages: Pearson {cmp['pearson']:.5f} | "
          f"max |diff| {cmp['max_abs_diff']:.3f} on scores of scale {cmp['score_scale']:.1f} | top-1 agreement {cmp['top1_agreement']:.0%}")
    print("REPRODUCES THE EARLIER RUN" if REPRO_OK else
          "DOES NOT REPRODUCE. Do not run the next cell: check the pins in section 2 and MAX_VISUAL_TOKENS/DPI.")

## 7 · The full run

Safe to interrupt and rerun: finished pages are skipped. Every `CKPT_EVERY` pages the checkpoint and the timing log are
copied to the Drive run folder. **After a disconnect:** rerun sections 1-4 and 6, then this cell. With `RESUME = True` (the
default) it picks up the newest checkpoint on Drive by itself. Without Drive it asks you to upload one.

In [ ]:
assert REPRO_OK, "the reproduction gate in section 6 did not pass"

CKPT_DIR = Path("/content/ckpt"); CKPT_DIR.mkdir(exist_ok=True)
EMB_DIR = Path("/content/embeddings"); EMB_DIR.mkdir(exist_ok=True)
EMB_DRIVE = RUN / "embeddings" if DRIVE else None
SESSION = time.strftime("%Y%m%d-%H%M%S")

scores = np.zeros((len(qids), len(page_keys)), dtype=np.float32)
done = np.zeros(len(page_keys), dtype=bool)
token_counts = np.zeros(len(page_keys), dtype=np.int32)

if RESUME:
    found = sorted(RUN.glob("colvec_ckpt_*.npz"))
    if not found and not DRIVE:
        up = files.upload()
        found = [Path(sorted(n for n in up if n.endswith(".npz"))[-1])]
    if found:
        ck = np.load(found[-1], allow_pickle=False)
        assert str(ck["fingerprint"]) == FINGERPRINT, "that checkpoint is from a different bundle"
        assert list(ck["keys"]) == page_keys and list(ck["qids"]) == qids, "page or query order differs from the checkpoint"
        scores, done, token_counts = ck["scores"], ck["done"], ck["tokens"]
        if (RUN / "timings.jsonl").exists():     # earlier sessions' per-page timings, up to that checkpoint
            lines = (RUN / "timings.jsonl").read_text().splitlines() + TIMINGS.read_text().splitlines()
            TIMINGS.write_text("".join(l + "\n" for l in dict.fromkeys(l for l in lines if l.strip())))
        print(f"resumed from {found[-1].name}: {int(done.sum())}/{len(page_keys)} pages already scored")
    else:
        print(f"no checkpoint in {RUN}: starting from the first page")

def write_checkpoint(n_done):
    path = CKPT_DIR / f"colvec_ckpt_{n_done:05d}.npz"
    np.savez_compressed(path, scores=scores, done=done, tokens=token_counts, keys=np.array(page_keys),
                        qids=np.array(qids), fingerprint=FINGERPRINT, model=MODEL_ID)
    if to_drive(path) and to_drive(TIMINGS):
        for old in sorted(RUN.glob("colvec_ckpt_*.npz"))[:-2]:     # keep the newest two on Drive
            old.unlink(missing_ok=True)
    return path

shard, shard_bytes, shard_no = {}, 0, 0
def flush_shard():
    global shard, shard_bytes, shard_no
    if shard:
        path = EMB_DIR / f"colvec_embeddings_{SESSION}_{shard_no:03d}.pt"   # session in the name: a resume never overwrites a shard
        torch.save(shard, path)
        if EMB_DRIVE and to_drive(path, EMB_DRIVE):
            path.unlink()
        shard, shard_bytes, shard_no = {}, 0, shard_no + 1

started, todo = perf_counter(), int((~done).sum())
n_new = 0
for d in documents:
    with fitz.open(BENCH / d["path"]) as pdf:
        for p in range(len(pdf)):
            key = f"{d['doc_id']}#page={p}"; j = col_of[key]
            if done[j]:
                continue
            col, e, tm = encode_page(pdf, p)
            scores[:, j], done[j], token_counts[j] = col, True, tm["tokens"]
            log_timing("colvec_page", "page", 1, tm["render_seconds"] + tm["encode_seconds"] + tm["score_seconds"],
                       unit_id=key, **tm)
            if SAVE_EMBEDDINGS:
                shard[key] = e.cpu(); shard_bytes += e.numel() * 2
                if shard_bytes >= SHARD_BYTES:
                    flush_shard()
            n_new += 1
            if n_new % 50 == 0:
                rate = (perf_counter() - started) / n_new
                print(f"  {int(done.sum())}/{len(page_keys)}  {rate:.2f}s/page  ETA {(todo - n_new) * rate / 60:.0f} min", flush=True)
            if n_new % CKPT_EVERY == 0:
                write_checkpoint(int(done.sum()))
if SAVE_EMBEDDINGS:
    flush_shard()
log_timing("colvec_corpus_wall", "corpus", n_new, perf_counter() - started)
final_ckpt = write_checkpoint(int(done.sum()))
print(f"done: {int(done.sum())}/{len(page_keys)} pages, {perf_counter() - started:.0f}s this session")
assert done.all(), f"{int((~done).sum())} pages not scored"

## 8 · Export, and the text to paste back

Writes `colvec_result.zip` (what the local scripts read) and `PASTE_ME.md` (a few KB: settings, the reproduction check and a
timing summary) to the Drive run folder, and prints `PASTE_ME.md` below so you can copy it straight from this page.

In [ ]:
OUT = Path("/content/colvec_result"); OUT.mkdir(exist_ok=True)

order = sorted(range(len(page_keys)), key=lambda i: page_keys[i])       # columns sorted by unit id, as the earlier export
keys_sorted = [page_keys[i] for i in order]
scores_sorted = scores[:, order]
assert scores_sorted.shape == (len(qids), len(page_keys)) and np.isfinite(scores_sorted).all()

np.save(OUT / "colvec_scores.npy", scores_sorted)
json.dump(keys_sorted, open(OUT / "colvec_keys.json", "w"))
json.dump(qids, open(OUT / "colvec_qids.json", "w"))
meta = {"model": MODEL_ID, "dpi": DPI, "max_visual_tokens": MAX_VISUAL_TOKENS, "load_8bit": LOAD_8BIT,
        "dtype": str(DTYPE), "dim": DIM, "n_pages": len(page_keys), "n_queries": len(qids),
        "bundle_fingerprint": FINGERPRINT, "smoke": manifest["smoke"], "gpu": gpu.name, "env": ENV,
        "mean_tokens_per_page": float(token_counts.mean()), "max_tokens_per_page": int(token_counts.max()),
        "batch_size": 1, "embeddings_saved": SAVE_EMBEDDINGS}
json.dump(meta, open(OUT / "colvec_meta.json", "w"), indent=1)
json.dump({k: int(t) for k, t in zip(page_keys, token_counts)}, open(OUT / "page_token_counts.json", "w"))
shutil.copy("/content/query_embeddings.pt", OUT / "query_embeddings.pt")

# A page re-encoded after a disconnect is logged twice: keep its last record.
rows = [json.loads(l) for l in TIMINGS.read_text().splitlines() if l.strip()]
last_page = {r["unit_id"]: i for i, r in enumerate(rows) if r["stage"] == "colvec_page"}
rows = [r for i, r in enumerate(rows) if r["stage"] != "colvec_page" or last_page[r["unit_id"]] == i]
(OUT / "timings.jsonl").write_text("".join(json.dumps(r) + "\n" for r in rows))

def timing_summary(rows):
    """Per stage: how many records and units, total seconds, and the per-unit spread (so a stall is visible, not averaged away)."""
    out = {}
    for stage in dict.fromkeys(r["stage"] for r in rows):
        rs = [r for r in rows if r["stage"] == stage]
        per = np.array([r["seconds"] / max(r["n"], 1) for r in rs])
        s = {"unit": rs[0]["unit"], "records": len(rs), "units": int(sum(r["n"] for r in rs)),
             "total_seconds": round(float(sum(r["seconds"] for r in rs)), 2), "per_unit_mean": round(float(per.mean()), 4),
             "per_unit_p50": round(float(np.percentile(per, 50)), 4), "per_unit_p95": round(float(np.percentile(per, 95)), 4)}
        for k in sorted({k for r in rs for k in r if k.endswith("_seconds")}):
            s[f"mean_{k}"] = round(float(np.mean([r[k] for r in rs if k in r])), 4)
        out[stage] = s
    return out

summary = timing_summary(rows)
json.dump(summary, open(OUT / "timings_summary.json", "w"), indent=1)
full = compare_to_reference(scores_sorted, keys_sorted, qids)
json.dump(full, open(OUT / "reference_check.json", "w"), indent=1)
if full:
    print(f"vs the earlier run over {full['n_queries']} queries x {full['n_pages']} pages: Pearson {full['pearson']:.5f}, "
          f"top-1 agreement {full['top1_agreement']:.1%}, mean |diff| {full['mean_abs_diff']:.4f}")

paste = "\n".join([
    f"# colvec result, bundle {FINGERPRINT}{' (SMOKE)' if manifest['smoke'] else ''}",
    f"written {time.strftime('%Y-%m-%d %H:%M')} | {int(done.sum())}/{len(page_keys)} pages | "
    f"{sum(r['stage'] == 'colvec_model_load' for r in rows)} session(s)",
    "", "## colvec_meta.json", "```json", json.dumps(meta, indent=1), "```",
    "", "## reference_check.json", "```json", json.dumps(full, indent=1), "```",
    "", "## timings_summary.json", "```json", json.dumps(summary, indent=1), "```", ""])
(OUT / "PASTE_ME.md").write_text(paste)

shutil.make_archive("/content/colvec_result", "zip", OUT)
print("wrote /content/colvec_result.zip", round(Path("/content/colvec_result.zip").stat().st_size / 1e6, 1), "MB")
on_drive = {f.name: to_drive(f) if DRIVE else None for f in (Path("/content/colvec_result.zip"), OUT / "PASTE_ME.md")}
print("\n".join(f"  on Drive: {p}" for p in on_drive.values() if p))

In [ ]:
if not on_drive["colvec_result.zip"]:        # no Drive, or the Drive copy failed: the browser is the only way out
    files.download("/content/colvec_result.zip")
print("=" * 30, "copy everything below this line into the chat", "=" * 30)
print((OUT / "PASTE_ME.md").read_text())

## 9 · Optional: raw embeddings

Only if `SAVE_EMBEDDINGS = True`. Each shard is a `torch.save` dict `{unit_id: fp16 tensor (tokens, 640)}` of about 1 GB. With Drive
they are already in `<run folder>/embeddings/` (about 11 GB in total, **more than a free 15 GB Drive holds**). A page encoded just before
a disconnect can sit in two shards; the values are the same. Nothing in the arms needs them: they exist so a *new query set* can be
scored without a GPU.

In [ ]:
if not SAVE_EMBEDDINGS:
    print("SAVE_EMBEDDINGS was False: no embeddings were kept.")
elif DRIVE:
    shards = sorted(EMB_DRIVE.glob("colvec_embeddings_*.pt"))
    print(f"{len(shards)} shards, {sum(s.stat().st_size for s in shards) / 1e9:.1f} GB in {EMB_DRIVE}")
else:
    for shard_file in sorted(EMB_DIR.glob("colvec_embeddings_*.pt")):
        print("downloading", shard_file.name, round(shard_file.stat().st_size / 1e9, 2), "GB")
        files.download(str(shard_file))